In [1]:
import os

In [2]:
%pwd

'd:\\krishnaik_proj\\end-to-end-ml-project-with-mlflow\\research'

In [4]:
os.chdir("../")

In [5]:
%pwd

'd:\\krishnaik_proj\\end-to-end-ml-project-with-mlflow'

In [8]:
import pandas as pd
df = pd.read_csv("artifacts\data_ingestion\winequality-red.csv")
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [11]:
list(df.columns)

['fixed acidity',
 'volatile acidity',
 'citric acid',
 'residual sugar',
 'chlorides',
 'free sulfur dioxide',
 'total sulfur dioxide',
 'density',
 'pH',
 'sulphates',
 'alcohol',
 'quality']

In [15]:
str(df['quality'].dtype)

'int64'

In [17]:
schema = {}

for col in df.columns:
    schema[col] = str(df[col].dtype)


In [18]:
import yaml

with open('schema.yaml','w') as f:
    yaml.dump({"COLUMNS":schema},f)

In [6]:
from dataclasses import dataclass
from pathlib import Path


In [37]:
#entity
@dataclass(frozen=True)
class DataValidationConfig:
    root_dir: Path
    unzip_data_dir: Path
    STATUS_FILE: str
    all_schema: dict


In [38]:
from mlproject.utils.common import read_yaml, create_directories
from mlproject.constants import *


In [39]:
#configuration manager

class ConfigurationManager:
    def __init__(
            self,
            config_filepath = CONFIG_FILE_PATH,
            schema_filepath = SCHEMA_FILE_PATH,
            params_filepath = PARAMS_FILE_PATH 
    ):
        self.config = read_yaml(config_filepath)
        self.schema = read_yaml(schema_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_validation_config(self)-> DataValidationConfig:
        config = self.config.data_validation
        schema = self.schema.COLUMNS

        create_directories([config.root_dir])

        data_validation_config = DataValidationConfig(
            root_dir = config.unzip_data_dir,
            STATUS_FILE = config.STATUS_FILE,
            unzip_data_dir = config.unzip_data_dir,
            all_schema = schema
        )
        return data_validation_config
            

In [40]:
import os
from mlproject import logger

In [45]:
# components

class DataValidation:
    def __init__(self,config:DataValidationConfig):
        self.config = config

    def validate_all_columns(self)->bool:
        try:
            data = pd.read_csv(self.config.unzip_data_dir)
            validation_status = all(
                col in self.config.all_schema.keys() for col in data.columns
            )
            with open(self.config.STATUS_FILE,"w") as f:
                f.write(f"validation_status: {validation_status}")
            logger.info(f"validation_status: {validation_status}")
        except Exception as e:
            logger.exception(e)
            raise e

In [ ]:
try:
    config = ConfigurationManager()
    data_validation_config = config.get_data_validation_config()
    data_validation = DataValidation(config=data_validation_config)
    data_validation.validate_all_columns()
except Exception as e:
    raise e
    

[2026-03-05 21:36:48,314: INFO: common: yaml file:(path_to_yaml) loaded sucesssfully]
[2026-03-05 21:36:48,324: INFO: common: yaml file:(path_to_yaml) loaded sucesssfully]
[2026-03-05 21:36:48,327: INFO: common: yaml file:(path_to_yaml) loaded sucesssfully]
[2026-03-05 21:36:48,332: INFO: common: created directory at:artifacts]
[2026-03-05 21:36:48,337: INFO: common: created directory at:artifacts/data_validation]
[2026-03-05 21:36:48,374: INFO: 2601172562: validation_status: True]
